# Getting Trustworthy Data: Collection, Point-in-Time Sets, Quality, Labels and Privacy

Notebook for **Chapter 2** of the *ds* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ds/ch02.html

Tip: change a number, predict the output, then run the cell again.

## 2.1 Data Sources and Point-in-Time Training Sets

In [ ]:
import sqlite3
from ds_common import *

c, o = make_data()
con = sqlite3.connect(":memory:")
c.assign(signup_date=c.signup_date.dt.strftime("%Y-%m-%d")).to_sql("customers", con, index=False)
o.assign(order_ts=o.order_ts.dt.strftime("%Y-%m-%d")).to_sql("orders", con, index=False)

sql = """
WITH base AS (
  SELECT c.customer_id, c.plan, c.channel,
         CAST(julianday(:T) - julianday(c.signup_date) AS INT) AS tenure_days
  FROM customers c WHERE c.signup_date < :T
),
past AS (
  SELECT customer_id,
         COUNT(*) AS orders_90d,
         SUM(amount) AS spend_90d,
         CAST(julianday(:T) - julianday(MAX(order_ts)) AS INT) AS recency_days
  FROM orders
  WHERE order_ts <  :T
    AND order_ts >= date(:T, '-90 day')
  GROUP BY customer_id
),
fut AS (
  SELECT DISTINCT customer_id FROM orders
  WHERE order_ts >= :T AND order_ts < date(:T, '+60 day')
)
SELECT b.*, p.orders_90d, p.spend_90d, p.recency_days,
       CASE WHEN f.customer_id IS NULL THEN 1 ELSE 0 END AS churn
FROM base b
JOIN past p USING (customer_id)          -- inner join = active customers only
LEFT JOIN fut f USING (customer_id)
"""
d = pd.read_sql(sql, con, params={"T": "2025-01-01"})
print(d.shape, round(d.churn.mean(), 3))
print(d.head(3).round(1).to_string())

(2562, 8) 0.254
   customer_id  plan   channel  tenure_days  orders_90d  spend_90d  recency_days  churn
0            1   pro  referral          666           3     2403.8             8      0
1            3  free   organic          254           8     3828.1            37      1
2            6   pro       ads          105           5     4034.3            34      0


In [ ]:
leaky = pd.read_sql("SELECT customer_id, MAX(order_ts) AS last_order FROM orders GROUP BY customer_id", con)
leaky["days_to_last"] = (pd.to_datetime(leaky.last_order) - pd.Timestamp("2025-01-01")).dt.days
dd = d.merge(leaky, on="customer_id")
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

def auc_for(cols, data):
    tr, te = train_test_split(data, test_size=0.3, random_state=0, stratify=data.churn)
    m = LogisticRegression(max_iter=2000).fit(tr[cols], tr.churn)
    return round(roc_auc_score(te.churn, m.predict_proba(te[cols])[:, 1]), 3)

print("honest AUC :", auc_for(["orders_90d", "recency_days", "spend_90d"], dd))
print("leaky  AUC :", auc_for(["orders_90d", "recency_days", "spend_90d", "days_to_last"], dd))

honest AUC : 0.882
leaky  AUC : 0.972


## 2.2 Data Quality Checks Before Any Modelling

In [ ]:
bad = o.copy()
dup = bad.sample(300, random_state=1)
bad = pd.concat([bad, dup], ignore_index=True)                    # 300 duplicate rows
bad.loc[bad.sample(40, random_state=2).index, "amount"] = -99.0   # impossible values
bad.loc[bad.sample(25, random_state=3).index, "order_ts"] = pd.Timestamp("2030-01-01")
bad.loc[bad.sample(150, random_state=4).index, "category"] = None

def quality_report(df, name="orders"):
    r = {}
    r["rows"] = len(df)
    r["dup_order_id"] = int(df.order_id.duplicated().sum())
    r["null_category_pct"] = round(df.category.isna().mean() * 100, 2)
    r["amount<=0"] = int((df.amount <= 0).sum())
    r["order_ts_in_future"] = int((df.order_ts > END_DATE).sum())
    r["orphan_customer_ids"] = int((~df.customer_id.isin(c.customer_id)).sum())
    return pd.Series(r, name=name)
print(pd.concat([quality_report(o, "clean"), quality_report(bad, "damaged")], axis=1).to_string())

                        clean   damaged
rows                 151719.0  152019.0
dup_order_id              0.0     300.0
null_category_pct         0.0       0.1
amount<=0                 0.0      40.0
order_ts_in_future        0.0      25.0
orphan_customer_ids       0.0       0.0


## 2.3 Labels and Ground Truth

## 2.4 Privacy and Ethics in Data Use

In [ ]:
import hashlib, hmac
SECRET = b"demo-secret-not-for-real-use"      # in real life: stored in a vault, never in code
c["cust_token"] = [hmac.new(SECRET, str(i).encode(), hashlib.sha256).hexdigest()[:10]
                   for i in c.customer_id]
c["age_band"] = pd.cut(c.age, [17, 25, 35, 50, 70], labels=["18-25", "26-35", "36-50", "51-70"])
print(c[["customer_id", "cust_token", "age_band"]].head(3).to_string(index=False))
kc = c.groupby(["city", "age_band"], observed=True).size()
print("smallest group (k) for city x age_band:", kc.min())
kc2 = c.groupby(["city", "age", "plan", "channel"], observed=True).size()
print("k for city x exact age x plan x channel:", kc2.min(), " groups of size 1:", int((kc2 == 1).sum()))

 customer_id cust_token age_band
           1 b1f22c3870    18-25
           2 fc2a3076b3    36-50
           3 797f02a27e    26-35
smallest group (k) for city x age_band: 15
k for city x exact age x plan x channel: 1  groups of size 1: 226


## Chapter recap

## Practice set

Problem 2.P.1 (easy) Which date filter belongs in a feature for prediction at T: `ts <= T + 30`, `ts < T`, or `ts >= T`? **Answer:** `ts < T`.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ds/ch02.html